### Analyse univariée des variables

Nous allons etudier notre base de données "df_imputation" après les traitements effectués

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df_imputation_provisoire = pd.read_csv("data/df_imputation.csv")

In [ ]:
print(df_imputation_provisoire.shape)
print(df_imputation_provisoire.columns)

In [ ]:
missing = pd.DataFrame({
    "nb_manquants": df_imputation_provisoire.isna().sum(),
    "pct_manquants": (df_imputation_provisoire.isna().mean() * 100).round(2)
})

missing.sort_values("nb_manquants", ascending=False)

Certaines variables socio-économiques, telles que l’indice de Gini ou la structure des revenus, présentent un intérêt théorique élevé pour expliquer le taux d’équipement en véhicules électriques.
Toutefois, leur taux de valeurs manquantes étant supérieur à 85%, leur utilisation aurait conduit à une réduction significative de la taille de l’échantillon et à un risque de biais de sélection.
Elles ont donc été exclues de l’analyse principale, mais peuvent faire l’objet d’analyses complémentaires sur un sous-échantillon restreint.

In [ ]:
df_imputation = df_imputation_provisoire.copy()
df_imputation = df_imputation.drop(columns=vars_to_drop_1)

In [ ]:
print(df_imputation.shape)
print(df_imputation.columns)

Analysons notre variable cible "taux_equipement_ve"

In [ ]:
target = "taux_equipement_ve"
print(df_imputation[target].describe())

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_imputation[target], bins=50)
plt.title("Distribution du taux d'équipement en véhicules électriques")
plt.xlabel("Taux d'équipement VE")
plt.ylabel("Effectif")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_imputation[target])
plt.title("Boxplot du taux d'équipement VE")
plt.xlabel("Taux d'équipement VE")
plt.show()

Le boxplot met en évidence une distribution fortement asymétrique à droite du taux d’équipement en véhicules électriques.
La majorité des communes présentent des taux faibles, tandis qu’un nombre limité d’observations affiche des valeurs élevées, générant de nombreux points extrêmes.
Ces valeurs élevées ne traduisent pas nécessairement des anomalies, mais peuvent s’expliquer par des effets de petite taille de population ou par des spécificités territoriales.
Afin de limiter l’influence de ces observations sur la modélisation, une transformation logarithmique de la variable cible sera envisagée.

In [ ]:
#df_imputation["log_taux_equipement_ve"] = np.log1p(df_imputation["taux_equipement_ve"])

## Analyse des variables explicatives

In [ ]:
# Variables à exclure (identifiants / non pertinentes)
cols_to_exclude = ["CODGEO", "Libellé géographique", target]

# Variables numériques explicatives
num_cols = [
    col for col in df_imputation.select_dtypes(include=["number"]).columns
    if col not in cols_to_exclude
]

# Variables qualitatives
cat_cols = [
    col for col in df_imputation.select_dtypes(include=["object"]).columns
    if col not in ["CODGEO", "Libellé géographique"]
]

print("Variables numériques :", num_cols)
print("\nVariables qualitatives :", cat_cols)

Regardons si des variables numériques ont peu de modalités alors nous pouvons les considérer comme qualitatives :

In [ ]:
df_imputation.nunique().sort_values()

Nous gardons les variables ainsi

Analyse univariée des variables numériques

In [ ]:
desc = df_imputation[num_cols].describe().T

desc["missing_pct"] = df_imputation[num_cols].isna().mean() * 100
desc["skewness"] = df_imputation[num_cols].skew()

display(desc.sort_values("missing_pct", ascending=False))

La base présente des distributions très asymétriques avec de nombreux zéros et quelques valeurs extrêmes, ce qui rend les moyennes peu représentatives. Cela traduit un fort effet de taille entre petites communes et grandes villes, qui risque de dominer l’analyse. Les variables liées aux infrastructures sont très peu renseignées (souvent nulles), tandis que les variables socio-économiques comportent près de 45 % de valeurs manquantes. Dans l’ensemble, la base nécessite des transformations (log, ratios) et une attention particulière aux biais liés à la taille et aux données manquantes avant toute modélisation.

In [ ]:
for col in num_cols:
    plt.figure()
    plt.hist(df_imputation[col].dropna(), bins=40)
    plt.title(f"Distribution de {col}")
    plt.xlabel(col)
    plt.ylabel("Effectif")
    plt.show()

### Analyse univariée des variables qualitatives

In [ ]:
for col in cat_cols:
    print(f"\n--- {col} ---")
    display(df_imputation[col].value_counts(dropna=False))

Cette variable risque d'être retravaillée car beaucoup de NA et importance avec la variable cible

### Analyses bivariées avec la cible

Variables numériques vs cible

In [ ]:
for col in num_cols:
    plt.figure()
    sns.regplot(x=df_imputation[col], y=df_imputation[target], scatter_kws={"alpha": 0.3})
    plt.title(f"Relation entre {col} et {target}")
    plt.show()

Nous pouvons constater quelques relations, comme Mediane et le taux d'équipement de véhicules électriques. En effet, une relation linéaire semble être envisageable.

Variables qualitatives vs cible

In [ ]:
for col in cat_cols:
    top = df_imputation[col].value_counts().head(10).index
    df_temp = df_imputation[df_imputation[col].isin(top)]
    
    plt.figure()
    sns.boxplot(x=df_temp[col], y=df_temp[target])
    plt.title(f"{target} selon principales modalités de {col}")
    plt.xticks(rotation=45)
    plt.show()

### Corrélations

In [ ]:
corr = df_imputation[[target] + num_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.show()

Ce que nous enseigne la matrice de corrélation :
Le taux VE dépend surtout :
- de la richesse ( plus une ville est riche, plus elle semble compter des VE )
- du volume de véhicules ( Plus une ville compte de véhicules, plus elle semble compter des VE )

Les infrastructures semblent peu expliquer directement, elles suivent la demande.

Des variables sont redondentes donc peuvent être éliminées.

Corrélations avec la cible

In [ ]:
corr_target = corr[target].sort_values(ascending=False)
display(corr_target)

Nous pouvons voir une forme d'explication avec certains groupes de variables comme celles socio démographiques. Un tri va être opéré pour une meilleure lisibilité.

Multicolinéarité

In [ ]:
corr_expl = df_imputation[num_cols].corr().abs()

upper = corr_expl.where(np.triu(np.ones(corr_expl.shape), k=1).astype(bool))

high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "var1", "level_1": "var2", 0: "corr"})
    .sort_values("corr", ascending=False)
)

display(high_corr[high_corr["corr"] > 0.8])

Nous choisissons une variable représentative des groupes de variables mis en évidence :

In [ ]:
vars_to_drop = [
    "NB_VP",
    "NB_VP_RECHARGEABLES_EL",
    "[DISP] Nbre de ménages fiscaux",
    "pct_type_2",
    "pct_type_ef",
    "part_charge_rapide"
]

df_clean = df_imputation.drop(columns=vars_to_drop, errors="ignore")

La variable cible étant construite comme un ratio entre le nombre de véhicules électriques et le nombre total de véhicules, les variables utilisées pour sa construction ont été exclues de l’analyse afin d’éviter tout biais de fuite d’information.
Leur inclusion aurait conduit à une relation mécanique avec la variable cible, faussant ainsi l’interprétation des résultats.

In [ ]:
df_clean

In [ ]:
missing_pct = df_clean.isna().mean() * 100
display(missing_pct.sort_values(ascending=False))

In [ ]:
df_clean.to_csv("data/df_clean.csv", index=False)